# Objective
## The main objectives of this project are:

+ To build an automated resume screening pipeline using Generative AI techniques
+ To extract key skills, experience, and tools from resumes
+ To compare extracted data with job requirements for matching
+ To generate a quantitative fit score (0–100) for each candidate
+ To provide clear and explainable reasoning for the assigned score
+ To implement a modular LLM-based workflow using prompt engineering and chaining
+ To understand real-world application of AI in recruitment systems

In [1]:
!pip install langchain langchain-openai langchain-community langsmith openai

In [2]:
!pip install requests==2.32.5

# Step 2

In [3]:
pip install -U langchain langchain-openai openai

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.2 MB ? eta -:--:--
   ------------------------------------ --- 1.0/1.2 MB 2.9 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 2.8 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.31.0
    Uninstalling openai-2.31.0:
      Successfully uninstalled openai-2.31.0
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install requests==2.32.5

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "your hf key here"

os.environ["LANGCHAIN_API_KEY"] = "your api key"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening"

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_huggingface import HuggingFaceEndpoint

In [18]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

hf_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=350
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'Deepseek

# Step 3

In [19]:
job_description = """
Looking for a Data Scientist with skills in Python, Machine Learning,
Deep Learning, SQL, and Data Analysis
"""

strong_resume = """
Data Scientist with 5 years experience in Python, Machine Learning,
Deep Learning, SQL, NLP, and Data Analysis.
"""

average_resume = """
Data Analyst with 2 years experience in Python, SQL, Excel,
and basic Machine Learning.
"""

weak_resume = """
Fresher with knowledge of MS Office and basic programming.
"""

# Step 4

In [20]:
extract_prompt = PromptTemplate.from_template("""
Extract skills from resume.

Resume:
{resume}
""")

extract_chain = extract_prompt | llm

In [21]:
job_skills = job_description.lower().split(",")

def match_logic(data):
    resume = data["skills"].lower()

    matched = [s for s in job_skills if s.strip() in resume]
    missing = [s for s in job_skills if s.strip() not in resume]

    return {
        "matched": matched,
        "missing": missing
    }

match_chain = RunnableLambda(match_logic)

In [22]:
def score_logic(data):
    total = len(job_skills)
    score = int((len(data["matched"]) / total) * 100)

    return {
        **data,
        "score": score
    }

score_chain = RunnableLambda(score_logic)

In [23]:
def explain_logic(data):
    return f"""
Score: {data['score']}

Matched: {data['matched']}
Missing: {data['missing']}

Candidate suitability based on skill overlap.
"""

explain_chain = RunnableLambda(explain_logic)

In [24]:
def pipeline(resume):

    step1 = extract_chain.invoke({"resume": resume})

    step2 = match_chain.invoke({"skills": step1})

    step3 = score_chain.invoke(step2)

    final = explain_chain.invoke(step3)

    print(final)

# STEP 5

In [25]:
resume = """
Data Scientist with 5 years experience in Python, Machine Learning,
Deep Learning, SQL, NLP, and Data Analysis.
"""

job_description = """
Looking for a Data Scientist with Python, Machine Learning,
Deep Learning, SQL, and Data Analysis experience.
"""

In [26]:
print("===== STRONG =====")
pipeline(strong_resume)

print("\n===== AVERAGE =====")
pipeline(average_resume)

print("\n===== WEAK =====")
pipeline(weak_resume)

===== STRONG =====


Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score: 80

Matched: [' machine learning', '\ndeep learning', ' sql', ' and data analysis\n']
Missing: ['\nlooking for a data scientist with skills in python']

Candidate suitability based on skill overlap.


===== AVERAGE =====

Score: 40

Matched: [' machine learning', ' sql']
Missing: ['\nlooking for a data scientist with skills in python', '\ndeep learning', ' and data analysis\n']

Candidate suitability based on skill overlap.


===== WEAK =====


Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score: 0

Matched: []
Missing: ['\nlooking for a data scientist with skills in python', ' machine learning', '\ndeep learning', ' sql', ' and data analysis\n']

Candidate suitability based on skill overlap.



# Step 6

In [16]:
if __name__ == "__main__":

    resumes = {
        "STRONG": strong_resume,
        "AVERAGE": average_resume,
        "WEAK": weak_resume
    }

    for name, resume in resumes.items():
        print(f"\n================ {name} CANDIDATE ================\n")

        try:
            result = pipeline(resume)

            # If result is dict-like
            if isinstance(result, dict):
                for key, value in result.items():
                    print(f"\n{key}:\n{value}")
            else:
                print(result)

        except Exception as e:
            print(f"Error processing {name} candidate:", str(e))

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



================ STRONG CANDIDATE ================


Score: 80

Matched: [' machine learning', '\ndeep learning', ' sql', ' and data analysis\n']
Missing: ['\nlooking for a data scientist with skills in python']

Candidate suitability based on skill overlap.

None

================ AVERAGE CANDIDATE ================



Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score: 40

Matched: [' machine learning', ' sql']
Missing: ['\nlooking for a data scientist with skills in python', '\ndeep learning', ' and data analysis\n']

Candidate suitability based on skill overlap.

None

================ WEAK CANDIDATE ================


Score: 0

Matched: []
Missing: ['\nlooking for a data scientist with skills in python', ' machine learning', '\ndeep learning', ' sql', ' and data analysis\n']

Candidate suitability based on skill overlap.

None


In [17]:
results = []

for name, resume in resumes.items():
    try:
        output = pipeline(resume)
        results.append((name, output))
        print(name, "processed successfully")
    except Exception as e:
        print(name, "failed:", e)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score: 80

Matched: [' machine learning', '\ndeep learning', ' sql', ' and data analysis\n']
Missing: ['\nlooking for a data scientist with skills in python']

Candidate suitability based on skill overlap.

STRONG processed successfully


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score: 40

Matched: [' machine learning', ' sql']
Missing: ['\nlooking for a data scientist with skills in python', '\ndeep learning', ' and data analysis\n']

Candidate suitability based on skill overlap.

AVERAGE processed successfully

Score: 0

Matched: []
Missing: ['\nlooking for a data scientist with skills in python', ' machine learning', '\ndeep learning', ' sql', ' and data analysis\n']

Candidate suitability based on skill overlap.

WEAK processed successfully
